# Feature engineering + model training

Builds features from the Silver table, trains a baseline fraud classifier,
and logs everything to MLflow — model, metrics, and parameters.

In [0]:
import mlflow
import mlflow.sklearn
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from mlflow.models import infer_signature
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    precision_recall_curve,
    average_precision_score,
)

CATALOG = "main"
SCHEMA = "fraud_project"

## Feature engineering (Gold layer)
The raw dataset's V1-V28 columns are already PCA-anonymized features, so the
main engineering work here is deriving transaction-pattern signals: how
unusual is this amount relative to recent activity, and when did it happen.

In [0]:
silver_df = spark.table(f"{CATALOG}.{SCHEMA}.silver_transactions")

# Rolling stats over a time-ordered window act as a stand-in for
# "transaction velocity" — how this amount compares to recent ones
time_window = Window.orderBy("Time").rowsBetween(-50, -1)

gold_features_df = (
    silver_df
    .withColumn("rolling_avg_amount", F.avg("Amount").over(time_window))
    .withColumn("rolling_std_amount", F.stddev("Amount").over(time_window))
    .withColumn(
        "amount_zscore",
        F.when(
            F.col("rolling_std_amount") > 0,
            (F.col("Amount") - F.col("rolling_avg_amount")) / F.col("rolling_std_amount"),
        ).otherwise(0.0),
    )
    .withColumn("is_night", F.when(F.col("transaction_hour").between(0, 5), 1).otherwise(0))
    .fillna({"rolling_avg_amount": 0.0, "rolling_std_amount": 0.0, "amount_zscore": 0.0})
)

(
    gold_features_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.gold_model_features")
)

print(f"Feature table row count: {gold_features_df.count()}")

{"ts": "2026-09-27 09:32:15.812", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvNDQ3NjI5ODc5MjcxMjc3MxABIAEyJDAxYTBlMjM0LWQ0ZjctNzZjMy05ZGM5LWYxMzQ1NDhkNzVkNDokMTk4MmE1YTAtZDc2ZC0zYTg0LTlkZGEtMDk3MWYwZmNkNDM4SgwIhMXj1QYQwIfHowFQAVgBYAFowqSz6/bEow0=.", "context": {}}
{"ts": "2026-09-27 09:32:15.812", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvNDQ3NjI5ODc5MjcxMjc3MxABIAEyJDAxYTBlMjM0LWQ0ZjctNzZjMy05ZGM5LWYxMzQ1NDhkNzVkNDokMTk4MmE1YTAtZDc2ZC0zYTg0LTlkZGEtMDk3MWYwZmNkNDM4SgwIhMXj1QYQwIfHowFQAVgBYAFowqSz6/bEow0=.", "context": {}}
{"ts": "2026-09-27 09:32:15.812", "level": "WARNING", "logger": "pyspark.sql.connect.logging", "msg": "Effective usage policy for this session is cct.Chpub3RlYm9va3MvNDQ3NjI5ODc5MjcxMjc3MxABIAEyJDAxYTBlMjM0LWQ0ZjctNzZjMy05ZGM5LWYxMzQ1NDhkNzVkNDokMTk4MmE1YTAtZDc2ZC0zYTg0LTlkZGEtMDk3MWYwZmNkNDM4

Feature table row count: 279950


## Train a baseline classifier
A random forest baseline, logged with MLflow. The dataset is heavily
imbalanced (~0.17% fraud), so precision/recall and average precision
matter far more here than accuracy.

In [0]:
pdf = gold_features_df.select(
    *[f"V{i}" for i in range(1, 29)],
    "Amount",
    "amount_zscore",
    "is_night",
    "is_fraud",
).toPandas()

feature_cols = [c for c in pdf.columns if c != "is_fraud"]
X = pdf[feature_cols]
y = pdf["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
mlflow.set_experiment(f"/Users/andrew.j.adjei@gmail.com/{SCHEMA}_fraud_baseline")

with mlflow.start_run(run_name="random_forest_baseline"):
    params = {
        "n_estimators": 200,
        "max_depth": 12,
        "class_weight": "balanced",  # important given the imbalance
        "random_state": 42,
        "n_jobs": -1,
    }
    mlflow.log_params(params)

    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    roc_auc = roc_auc_score(y_test, y_proba)
    avg_precision = average_precision_score(y_test, y_proba)

    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.log_metric("average_precision", avg_precision)

    print(classification_report(y_test, y_pred))
    print(f"ROC AUC: {roc_auc:.4f}")
    print(f"Average precision: {avg_precision:.4f}")

signature = infer_signature(X_train, model.predict(X_train))

mlflow.sklearn.log_model(
    model,
    artifact_path="model",
    signature=signature,
    input_example=X_train.iloc[:5],
    registered_model_name=f"{CATALOG}.{SCHEMA}.fraud_baseline_rf",
)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     55895
           1       0.87      0.72      0.79        95

    accuracy                           1.00     55990
   macro avg       0.94      0.86      0.89     55990
weighted avg       1.00      1.00      1.00     55990

ROC AUC: 0.9736
Average precision: 0.7946


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/09/27 09:35:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
🔗 View Logged Model at: https://dbc-38a66343-0fe1.cloud.databricks.com/ml/experiments/3956222057008302/models/m-f34660380f754150b4866

Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

🔗 Created version '2' of model 'main.fraud_project.fraud_baseline_rf': https://dbc-38a66343-0fe1.cloud.databricks.com/explore/data/models/main/fraud_project/fraud_baseline_rf/version/2?o=7474649385194050
